# General spin-spiral ordering vector

This file contains functions that create a spin-spiral structure based on a magnetic ordering vector and a normal-vector

The magnetization density satisfies:
$$\mathbf{m(r+r_i)} = R_{\hat n}(\mathbf{q\cdot r_i})\mathbf{m(r)}$$
(see the paper).

In [55]:
from ase.io import read
import numpy as np
from ase.build import make_supercell

In [56]:
def rotation_matrix(axis, theta):
    """
    Compute the rotation matrix for rotation around a given axis by theta radians.
    Uses quaternion representation for numerical stability.
    """
    axis = np.asarray(axis, dtype=float)
    axis = axis / np.linalg.norm(axis)  # Normalize the axis
    a = np.cos(theta / 2.0)
    b, c, d = -axis * np.sin(theta / 2.0)
    # Quaternion to rotation matrix
    R = np.array([
        [a*a + b*b - c*c - d*d, 2*(b*c - a*d), 2*(b*d + a*c)],
        [2*(b*c + a*d), a*a + c*c - b*b - d*d, 2*(c*d - a*b)],
        [2*(b*d - a*c), 2*(c*d + a*b), a*a + d*d - b*b - c*c]
    ])
    return R

def create_spin_spiral(atoms, primitive, Q, m0, n_hat, magnetic_symbols=None):
    """
    Create a spin-spiral magnetic structure.

    Parameters:
    - atoms: ASE Atoms object (supercell)
    - primitive: ASE Atoms object (primitive cell)
    - Q: Ordering vector in reciprocal lattice units of primitive cell (2D or 3D array-like)
    - m0: Initial/reference spin vector (3D array-like)
    - n_hat: Normal vector to the spin spiral plane (3D array-like, will be normalized)
    - magnetic_symbols: List of symbols for magnetic atoms (default: ['Mn'])

    Returns:
    - atoms: The Atoms object with magnetic moments set
    """
    if magnetic_symbols is None:
        magnetic_symbols = ['Mn']
    Q     = np.asarray(Q, dtype=float)
    m0    = np.asarray(m0, dtype=float)
    n_hat = np.asarray(n_hat, dtype=float)

    magmoms = np.zeros((len(atoms), 3))


    A     = primitive.cell[:2, :2]  # 2D lattice vectors from primitive cell
    A_inv = np.linalg.inv(A.T)      # Mapping to lattice coordinates


    for i, atom in enumerate(atoms):
        if atom.symbol not in magnetic_symbols:
            continue

        r_cart = atom.position[:2]  # Cartesian position in 2D plane
        n = A_inv @ r_cart  # Lattice coordinates in primitive basis

        # Phase angle: theta = 2*pi * dot(Q, n) for CLOCKWISE rotation (matching original)
        theta = 2 * np.pi * np.dot(Q[:2], n)
        R = rotation_matrix(n_hat, theta)
        magmoms[i] = R @ m0

    atoms.set_initial_magnetic_moments(magmoms)
    return atoms


Testing below

In [57]:
primitive = read("../1MnI2-1.cif")

name = primitive.get_chemical_formula(mode='metal')

#Define transformation matrix from primitive to magnetic cell
P = np.array([
    [2, 1, 0],
    [-1, 1, 0],
    [0, 0, 1]
])

supercell = make_supercell(primitive, P)

In [58]:
Q = np.array([1/3, 1/3,0])  # 2D ordering vector in primitive reciprocal units
m0 = np.array([0, 4.5, 0])   # Initial spin
n_hat = np.array([0, 0, 1])  # Normal vector (z-axis)
supercell = create_spin_spiral(supercell, primitive, Q, m0, n_hat)

In [59]:
from ase.visualize import view
view(supercell)

<Popen: returncode: None args: ['/home/runerce/Desktop/DFT-project-2026/.ven...>